### 2.1

在 `ababc` 中，以 `b` 开头的转移有 $b\to a$ 和 $b\to c$，各出现 1 次。

$$p(a|b)=\frac{1}{2},\qquad p(c|b)=\frac{1}{2}$$


In [1]:
# 2.2
import re
from collections import Counter

def preprocess_text(text, n):
    words = re.sub(r'[^a-z\s]', '', text.lower()).split()
    counts = Counter(words)
    vocab = {
        word: i for i, word in enumerate(
            sorted(counts, key=lambda x: (-counts[x], words.index(x)))
        )
    }
    features = [words[i:i+n] for i in range(len(words) - n + 1)]
    labels = [words[i+n] if i+n < len(words) else None
              for i in range(len(words) - n + 1)]
    return vocab, features, labels

vocab, features, labels = preprocess_text("The time machine!", 2)
print(vocab)
print(features)
print(labels)


{'the': 0, 'time': 1, 'machine': 2}
[['the', 'time'], ['time', 'machine']]
['machine', None]


### 3.1

令 $e_t=o_t-y_t$，则

$$\delta_t=\frac{\partial L}{\partial h_t}
=W_{oh}^Te_t+W_{hh}^T\delta_{t+1}$$

所以

$$\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^T\delta_t h_{t-1}^T$$

展开后：

$$\delta_t=\sum_{k=t}^T(W_{hh}^T)^{k-t}W_{oh}^T(o_k-y_k)$$

当 $W_{hh}$ 连乘后的值趋近 0 时梯度消失，数值不断增大时梯度爆炸。


In [2]:
# 3.2
import torch

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    h_t = torch.tanh(x_t @ W_hx + h_prev @ W_hh + b_h)
    return h_t, (x_t, h_prev, W_hx, W_hh, h_t)

def rnn_backward(dh_next, cache):
    x_t, h_prev, W_hx, W_hh, h_t = cache
    da = dh_next * (1 - h_t ** 2)
    dx_t = da @ W_hx.T
    dh_prev = da @ W_hh.T
    dW_hx = x_t.T @ da
    dW_hh = h_prev.T @ da
    db_h = da.sum(0)
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

torch.manual_seed(0)
x_t = torch.randn(2, 3)
h_prev = torch.randn(2, 4)
W_hx = torch.randn(3, 4)
W_hh = torch.randn(4, 4)
b_h = torch.randn(4)
dh_next = torch.randn(2, 4)

h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
grads = rnn_backward(dh_next, cache)
print("h_t:", h_t.shape)
print("梯度形状:", [x.shape for x in grads])


h_t: torch.Size([2, 4])
梯度形状: [torch.Size([2, 3]), torch.Size([2, 4]), torch.Size([3, 4]), torch.Size([4, 4]), torch.Size([4])]


### 4.1

第 1 层参数量：

$$2(HD+H^2+H)$$

第 2 到第 $L$ 层参数量：

$$2(L-1)(2H^2+H^2+H)$$

输出层参数量：

$$2HO+O$$

总参数量为

$$\boxed{2(HD+H^2+H)+2(L-1)(3H^2+H)+2HO+O}$$


In [3]:
# 4.2
import torch
import torch.nn as nn

class BiRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.rnn = nn.RNN(input_dim, hidden_dim, bidirectional=True)

    def forward(self, X):
        output, h = self.rnn(X)
        final_h = torch.cat((h[-2], h[-1]), dim=1)
        return output, final_h

X = torch.randn(6, 3, 4)
model = BiRNN(4, 5)
output, final_h = model(X)
print(output.shape)
print(final_h.shape)


torch.Size([6, 3, 10])
torch.Size([3, 10])


### 5.1

Skip-gram 负采样损失为

$$L=-\log\sigma(u_o^Tv_c)
-\sum_{k=1}^K\log\sigma(-u_{n_k}^Tv_c)$$

负样本从噪声分布中采样，常用

$$P_n(w)=\frac{f(w)^{3/4}}{\sum_{w'}f(w')^{3/4}}$$


In [4]:
# 5.2
import torch

def cbow_loss(context, target, W, W_out):
    h = W[context].mean(dim=1)
    logits = h @ W_out
    probs = torch.softmax(logits, dim=1)
    loss = -torch.log(probs[range(len(target)), target]).mean()
    return loss

torch.manual_seed(0)
V, d = 8, 5
context = torch.tensor([[0, 1, 3, 4], [2, 3, 5, 6]])
target = torch.tensor([2, 4])
W = torch.randn(V, d)
W_out = torch.randn(d, V)

loss = cbow_loss(context, target, W, W_out)
print("loss:", loss.item())


loss: 3.4364469051361084


### 6.1

先计算分数矩阵：

$$S=\frac{QK^T}{\sqrt{4}}=\frac{QK^T}{2},\qquad S\in\mathbb{R}^{2\times3}$$

再按行计算 softmax：

$$A=\operatorname{softmax}(S)$$

最后：

$$\boxed{O=AV=\operatorname{softmax}\left(\frac{QK^T}{2}\right)V}$$

输出形状为 $2\times5$。


In [5]:
# 6.2
import math
import torch
import torch.nn as nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        s, b, _ = x.shape
        x = x.reshape(s, b, self.num_heads, self.d_k)
        return x.permute(1, 2, 0, 3)

    def forward(self, X):
        s, b, _ = X.shape
        Q = self.split_heads(self.W_q(X))
        K = self.split_heads(self.W_k(X))
        V = self.split_heads(self.W_v(X))
        score = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        attention = torch.softmax(score, dim=-1)
        output = attention @ V
        output = output.permute(2, 0, 1, 3).reshape(s, b, -1)
        return self.W_o(output)

X = torch.randn(5, 3, 4)
mha = MultiHeadAttention()
output = mha(X)
print("输入:", X.shape)
print("输出:", output.shape)


输入: torch.Size([5, 3, 4])
输出: torch.Size([5, 3, 4])
